In [1]:
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import pickle
import pandas as pd
from tqdm.notebook import tqdm
from scipy.special import ndtri
from scipy.stats import norm
from scipy.stats import bootstrap
from sklearn.metrics import r2_score

from methyldl.deconvolution.evaluation import compute_deconvolution_metrics

%load_ext autoreload
%autoreload 2

In [2]:
ROOT_DATA_DIR = Path("/staging/leuven/stg_00118/methylDL/experiments/ExtendedProportions/pseudobulk/methylBert_Loyfer_205files_U25_attentionClassifier_hg38_dmr_ctype_label_mincpg_4_minlen_10_postfiltered_min_length_50_soft_labels_pooled_jakkard_no_data_leak_d041")
CALIBRATION_RESULTS_DIR = ROOT_DATA_DIR / "pseudobulk/fitted_deconvolvers_unifrorm_multinomial_all_top156features/callibration"

## Data loading

In [3]:
# Load test predictions for all deconvolvers and calibration methods
DECONVOLVER_NAMES = ["psls"]#["nnls", "psls", "xgb", "swn", "mlp"]
CALIBRATION_METHODS = [
    "uncalibrated",
    # "linear_clip01_normalize",
    "linear_clip0_normalize",
    "linear_simplex_projection",
    "vector_scaling",
]

predictions = {}  # {deconv_name: {method: test_pred array}}
target_proportions = None

for deconv_name in DECONVOLVER_NAMES:
    deconv_dir = CALIBRATION_RESULTS_DIR / f"{deconv_name}_calibrators_and_predictions"
    if not deconv_dir.exists():
        print(f"WARNING: {deconv_dir} not found, skipping")
        continue

    predictions[deconv_name] = {}

    # Uncalibrated predictions (also contains targets)
    uncalib_path = deconv_dir / "uncalibrated_predictions.npz"
    if uncalib_path.exists():
        data = np.load(uncalib_path)
        predictions[deconv_name]["uncalibrated"] = data["test_pred"]
        # Load target proportions (same across all deconvolvers)
        if target_proportions is None:
            target_proportions = data["test_target"]

    # Linear calibrated predictions
    for norm_method in ["clip0_normalize", "simplex_projection"]:#"clip01_normalize", 
        pred_path = deconv_dir / f"linear_{norm_method}_predictions.npz"
        if pred_path.exists():
            data = np.load(pred_path)
            predictions[deconv_name][f"linear_{norm_method}"] = data["test_pred"]

    # Vector scaling predictions
    vs_path = deconv_dir / "vector_scaling_predictions.npz"
    if vs_path.exists():
        data = np.load(vs_path)
        predictions[deconv_name]["vector_scaling"] = data["test_pred"]

print(f"Loaded predictions for deconvolvers: {list(predictions.keys())}")
for name, methods in predictions.items():
    print(f"  {name}: {list(methods.keys())}")
print(f"Target proportions shape: {target_proportions.shape}")

Loaded predictions for deconvolvers: ['psls']
  psls: ['uncalibrated', 'linear_clip0_normalize', 'linear_simplex_projection', 'vector_scaling']
Target proportions shape: (100000, 39)


## Radat plots of MSE for calibration methods

In [4]:
mse_per_ctype = {}
for deconv_name, deconv_res in predictions.items():
    mse_per_ctype[deconv_name] = {}
    for method_name, pred in deconv_res.items():
        mse = np.mean((pred - target_proportions) ** 2, axis=0)  # MSE per cell type
        mse_per_ctype[deconv_name][method_name] = mse

In [5]:
mse_per_ctype["psls"]

{'uncalibrated': array([5.04274137e-04, 2.69485487e-04, 2.87746308e-04, 8.05748174e-06,
        6.37155947e-05, 8.23403500e-06, 2.67536664e-04, 3.19450753e-05,
        1.18064837e-04, 3.15646572e-05, 5.62318621e-06, 1.82169069e-03,
        2.55441502e-04, 8.03762820e-04, 7.42374019e-06, 1.90002851e-05,
        2.40215137e-04, 1.75740419e-05, 6.38770026e-06, 1.71766641e-05,
        2.07242396e-05, 6.14779250e-05, 8.73339199e-05, 1.86196943e-05,
        5.01177106e-05, 2.67616030e-05, 1.56045485e-05, 1.15402270e-05,
        8.07157906e-04, 5.31391013e-06, 3.30787520e-05, 8.29580284e-06,
        3.14663259e-05, 7.60181642e-05, 2.16221081e-04, 1.35355835e-05,
        9.87071669e-06, 6.06520683e-04, 6.32738501e-06]),
 'linear_clip0_normalize': array([3.62494271e-04, 2.05241600e-04, 1.87174660e-04, 1.53418653e-05,
        4.67680191e-05, 6.71054880e-05, 1.36822140e-04, 3.26582588e-05,
        5.26139306e-05, 8.33397926e-05, 2.30994667e-05, 1.45001332e-03,
        2.22880380e-04, 9.07325455e-

In [ ]:
import json
import plotly.graph_objects as go

psls_mse = mse_per_ctype["psls"]

labels_dict_path = Path("../App/labels_dict.json")

with labels_dict_path.open("r", encoding="utf-8") as handle:
    labels_dict = json.load(handle)

n_ctypes = len(next(iter(psls_mse.values())))
ctype_labels = [labels_dict[str(index)] for index in range(n_ctypes)]

if len(ctype_labels) != 39:
    raise ValueError(f"Expected 39 cell types, found {len(ctype_labels)}")

mse_df = pd.DataFrame(psls_mse, index=ctype_labels)
mse_rank_df = mse_df.rank(axis=1, method="min", ascending=True)

# Invert ranks so that rank 1 maps to the outer radius
m = len(mse_rank_df.columns)
inverted_rank_df = m - mse_rank_df + 1

fig = go.Figure()
for method_name in inverted_rank_df.columns:
    method_ranks = inverted_rank_df[method_name].to_numpy()
    fig.add_trace(
        go.Scatterpolar(
            r=np.r_[method_ranks, method_ranks[0]],
            theta=ctype_labels + [ctype_labels[0]],
            mode="lines+markers",
            name=method_name,
        )
    )

# Show original rank values on the ticks (outermost tick = rank 1)
tickvals = list(range(1, m + 1))
ticktext = [str(m - v + 1) for v in tickvals]

fig.update_layout(
    title="PSLS MSE Rank per Cell Type by Calibration Method (rank 1 = outer)",
    polar={
        "radialaxis": {
            "visible": True,
            "dtick": 1,
            "range": [1, m],
            "tickvals": tickvals,
            "ticktext": ticktext,
        }
    },
    showlegend=True,
    width=950,
    height=850,
)

fig.show()

In [11]:
# Cluster cell types by their rank profiles and plot clustered radar
from sklearn.cluster import AgglomerativeClustering
import scipy.cluster.hierarchy as sch

# mse_rank_df: index=cell types, columns=methods
rank_matrix = mse_rank_df.values  # shape (n_ctypes, n_methods)

# Perform hierarchical clustering on rows (cell types)
n_clusters = 6  # choose a reasonable default
# Use current sklearn parameter names (no 'affinity')
clustering = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
row_labels = clustering.fit_predict(rank_matrix)

# Order cell types by hierarchical ordering
linkage_matrix = sch.linkage(rank_matrix, method='ward')
dendro = sch.dendrogram(linkage_matrix, no_plot=True)
row_order = dendro['leaves']

ordered_ctypes = [mse_rank_df.index[i] for i in row_order]
ordered_rank_df = mse_rank_df.loc[ordered_ctypes]

# Plot clustered radar
fig_cluster = go.Figure()
for method_name in ordered_rank_df.columns:
    vals = ordered_rank_df[method_name].to_numpy()
    fig_cluster.add_trace(
        go.Scatterpolar(
            r=np.r_[vals, vals[0]],
            theta=ordered_ctypes + [ordered_ctypes[0]],
            mode='lines+markers',
            name=method_name,
        )
    )

n_methods = len(mse_rank_df.columns)
fig_cluster.update_layout(
    title=f"Clustered PSLS MSE Rank per Cell Type (n_clusters={n_clusters})",
    polar={'radialaxis': {'visible': True, 'dtick': 1, 'range': [n_methods+0.5, 1]}},
    showlegend=True,
    width=1200,
    height=900,
)

fig_cluster.show()

In [12]:
# Boxplot of MSE per cell type: one box per calibration method
import json
import plotly.graph_objects as go
from pathlib import Path
import pandas as pd
import numpy as np

psls_mse = mse_per_ctype["psls"]

# Ensure we have labels
if 'ctype_labels' not in globals():
    labels_dict_path = Path("../App/labels_dict.json")
    with labels_dict_path.open("r", encoding="utf-8") as f:
        labels_dict = json.load(f)
    n_ctypes = len(next(iter(psls_mse.values())))
    ctype_labels = [labels_dict[str(i)] for i in range(n_ctypes)]

# Build dataframe rows=ctypes, columns=methods
mse_df = pd.DataFrame(psls_mse, index=ctype_labels)

fig = go.Figure()
for method in mse_df.columns:
    vals = mse_df[method].values
    fig.add_trace(go.Box(y=vals, name=method, boxmean=True))

fig.update_layout(
    title="PSLS MSE distribution across 39 cell types — one box per calibration method",
    yaxis_title="MSE",
    width=1000,
    height=600,
)

fig.show()